# Cell Embeddings Extraction from Single-Cell Foundation Models

This notebook extracts cell embeddings from multiple single-cell foundation models for downstream analysis. The supported models include:

- **scVI**: Variational autoencoder for single-cell RNA-seq data
- **Geneformer**: Transformer pretrained on single-cell transcriptomes
- **scGPT**: Generative pretrained transformer for single-cell analysis
- **UCE**: Universal Cell Embeddings using protein embeddings
- **xTrimoGene**: Foundation model for single-cell genomics
- **LangCell**: Combines cellular biology knowledge with language modeling
- **scBERT**: BERT-based model for single-cell analysis
- **scCello**: Contrastive learning approach for single-cell data
- **Harmony**: Batch correction and integration method

The extracted embeddings are saved as numpy arrays for use in downstream analysis tasks like clustering, classification, and visualization.

## 1. Import Libraries and Setup Environment

First, we import all necessary libraries and configure the environment for GPU usage and warning suppression.


In [6]:
import os
# Set CUDA device order for consistent GPU selection
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.chdir("../")
import sys
# Add custom modules to Python path
sys.path.insert(0, "./sc_foundation_evals")

import argparse
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scvi
import harmonypy as hm
import subprocess
import time
from functools import wraps

# Import model-specific forward functions
from sc_foundation_evals import (
    geneformer_forward, 
    scgpt_forward, 
    langcell_forward, 
    scbert_forward, 
    sccello_forward
)
from sc_foundation_evals import data

# Suppress warnings for cleaner output
import warnings
os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings("ignore")

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


## 2. Utility Functions

Define helper functions for model parameter calculation, resource monitoring, and data preprocessing.


In [7]:
def calculate_params(model):
    """
    Calculate the total number of parameters in a PyTorch model.
    
    Args:
        model: PyTorch model
        
    Returns:
        int: Total number of parameters
    """
    total_params = sum(
        param.numel() for param in model.parameters()
    )
    return total_params


def monitor_inference_resources(func):
    """
    Decorator to monitor the execution time of inference functions.
    
    Args:
        func: Function to be monitored
        
    Returns:
        Wrapped function with timing information
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"⏱️  Inference function '{func.__name__}' executed in {end_time - start_time:.2f} seconds")
        return result
    return wrapper

print("🛠️  Utility functions defined")

🛠️  Utility functions defined


## 3. Overall Configuration

Define all configuration parameters for different models and analysis options.

In [8]:
## 4. Overall Configuration (Default Parameters)
DEVICE = 'cuda'
DATA_FOLDER = '/mnt/nvme/extra_data/wujialu/scFM-Bench/data/datasets'
MODEL_FOLDER = '/mnt/nvme/extra_data/wujialu/scFM-Bench/data/weights'
DATASET_NAME = 'pancreas_scib'
DATASET_TYPE = 'reference'
REF_DATASET_NAME = 'Tabula_Sapiens_all'
SCVI_REF_PATH = '/mnt/nvme/extra_data/wujialu/scFM-Bench/output_test/Tabula_Sapiens_all/scVI/model.pt'
REF_BATCH_COL = None
BATCH_COL = None
LABEL_COL = None
GENE_COL = 'gene_symbols'
LAYER_KEY = 'counts'
OUTPUT_FOLDER = '/mnt/nvme/extra_data/wujialu/scFM-Bench/output_test'
BATCH_SIZE = 32
NUM_WORKERS = 1
DATA_IS_RAW = 1
NORMALIZE_TOTAL = 1e4
SEED = 7
ADATA_PATH = os.path.join(DATA_FOLDER, f"{DATASET_NAME}.h5ad")
REF_ADATA_PATH = os.path.join(DATA_FOLDER, f"{REF_DATASET_NAME}.h5ad")
OUTPUT_DIR = os.path.join(OUTPUT_FOLDER, DATASET_NAME, LAYER_KEY)
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

## 4. Data Preprocessing Functions

Functions for gene selection, data preprocessing, and format conversion required by different models.


In [9]:
def main_gene_selection(X_df, gene_list):
    """
    Rebuild the input data to select target genes and pad missing genes with zeros.
    
    This function ensures that the input data contains all genes required by the model,
    padding missing genes with zeros to maintain compatibility.
    
    Args:
        X_df (pd.DataFrame): Expression data with genes as columns
        gene_list (list): Target gene list required by the model
        
    Returns:
        tuple: (processed_df, missing_genes, var_metadata)
            - processed_df: DataFrame with target genes in correct order
            - missing_genes: List of genes that were padded with zeros
            - var_metadata: DataFrame tracking which genes were padded
    """
    # Find genes that need to be padded with zeros
    to_fill_columns = list(set(gene_list) - set(X_df.columns))
    
    # Create padding DataFrame with zeros for missing genes
    padding_df = pd.DataFrame(
        np.zeros((X_df.shape[0], len(to_fill_columns))), 
        columns=to_fill_columns, 
        index=X_df.index
    )
    
    # Concatenate original data with padding
    X_df = pd.DataFrame(
        np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
        index=X_df.index, 
        columns=list(X_df.columns) + list(padding_df.columns)
    )
    
    # Reorder columns to match target gene list
    X_df = X_df[gene_list]
    
    # Create metadata for tracking padded genes
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    
    return X_df, to_fill_columns, var


def preprocess_sc_data(adata, min_genes=25, min_cells=10):
    """
    Standard preprocessing pipeline for single-cell data.
    
    This function performs quality control filtering and normalization steps
    commonly used in single-cell analysis.
    
    Args:
        args: Configuration arguments
        adata: AnnData object with single-cell data
        min_genes (int): Minimum number of genes per cell for filtering
        min_cells (int): Minimum number of cells per gene for filtering
        
    Returns:
        AnnData: Preprocessed single-cell data
    """
    # Extract expression data from specified layer
    if LAYER_KEY == "X":
        if DATA_IS_RAW and adata.raw is not None:
            adata.X = adata.raw.X.copy()
            del adata.raw
            print("Copy raw counts of gene expressions from adata.raw.X")
    else:
        adata.X = adata.layers[LAYER_KEY].copy()
        print(f"Copy raw counts of gene expressions from adata.layers of {LAYER_KEY}")

    # Apply standard preprocessing if data is raw
    if DATA_IS_RAW:
        # Filter cells with too few genes
        sc.pp.filter_cells(adata, min_genes=min_genes) 
        print(f"After filter cells: {adata.X.shape}")
        
        # Filter genes expressed in too few cells
        sc.pp.filter_genes(adata, min_cells=min_cells)
        print(f"After filter genes: {adata.X.shape}")
        
        # Normalize to target sum and log-transform
        sc.pp.normalize_total(adata, target_sum=NORMALIZE_TOTAL)
        sc.pp.log1p(adata)

    return adata

print("📊 Data preprocessing functions defined")


📊 Data preprocessing functions defined


## 5. Model-Specific Inference Functions

### 5.1 scVI

scVI is a deep generative model for single-cell RNA sequencing data that uses variational autoencoders to learn low-dimensional representations while accounting for technical noise and batch effects.


In [10]:
@monitor_inference_resources
def run_scvi():
    """
    Extract cell embeddings using scVI model.
    
    scVI uses a variational autoencoder to learn cell representations that account for
    technical noise, dropout, and batch effects in single-cell RNA-seq data.
    
    Steps:
    1. Load and preprocess data
    2. Select highly variable genes
    3. Train scVI model
    4. Extract latent representations as cell embeddings
    """
    print("🧬 Starting scVI cell embedding extraction...")
    
    # Load and preprocess data
    adata = sc.read(ADATA_PATH)
    adata = preprocess_sc_data(adata)
    
    # Select highly variable genes for better performance
    sc.pp.highly_variable_genes(adata, flavor="seurat", subset=False, n_top_genes=N_HVG, batch_key=BATCH_COL)
    adata = adata[:, adata.var.highly_variable].copy()
    print(f"Data shape after HVG selection: {adata.X.shape}")
    
    # Setup scVI model with batch correction if specified
    scvi.model.SCVI.setup_anndata(adata, batch_key=BATCH_COL)
    
    # Create and train the model (ZINB loss by default)
    vae = scvi.model.SCVI(adata, n_layers=2, n_latent=30)
    vae.train(use_gpu=True)

    # Save the trained model
    vae.save(os.path.join(OUTPUT_DIR, "scvi"), overwrite=True)

    # Extract cell embeddings from the latent space
    cell_embeddings = vae.get_latent_representation()

    print(f"Cell embeddings shape: {cell_embeddings.shape}")  
    emb_file = f"cell_emb_wo_batch.npy" if BATCH_COL is None else f"cell_emb_{BATCH_COL}.npy"
    np.save(os.path.join(OUTPUT_DIR, "scvi", emb_file), cell_embeddings)
    print("✅ scVI embeddings saved successfully")




In [ ]:
@monitor_inference_resources
def run_scvi_surgery():
    """
    Extract cell embeddings using scVI surgery for query dataset integration.
    
    scVI surgery allows integrating new query data with a pre-trained reference model
    without retraining from scratch, enabling efficient batch correction.
    """
    print("🧬 Starting scVI surgery for query dataset...")
    
    query_adata = sc.read(ADATA_PATH)
    query_adata = preprocess_sc_data(query_adata)
    query_adata.obs[REF_BATCH_COL] = query_adata.obs[BATCH_COL]
    
    # Prepare query data for integration with reference model
    scvi.model.SCVI.prepare_query_anndata(query_adata, SCVI_REF_PATH)
    
    # Load and train query model
    scvi_query = scvi.model.SCVI.load_query_data(
        query_adata,
        SCVI_REF_PATH,
        use_gpu=True
    )
    scvi_query.train(use_gpu=True)
    scvi_query.save(os.path.join(OUTPUT_DIR, "scvi_surgery"), overwrite=True)
    
    # Extract embeddings
    cell_embeddings = scvi_query.get_latent_representation()
    print(f"Query cell embeddings shape: {cell_embeddings.shape}")  
    emb_file = f"cell_emb_wo_batch.npy" if BATCH_COL is None else "cell_emb.npy"
    np.save(os.path.join(OUTPUT_DIR, "scvi_surgery", emb_file), cell_embeddings)
    print("✅ scVI surgery embeddings saved successfully")

In [ ]:
N_HVG = 2000
if DATASET_TYPE == "query":
    run_scvi_surgery()
else:
    run_scvi()

🧬 Starting scVI cell embedding extraction...
Copy raw counts of gene expressions from adata.layers of counts
After filter cells: (16382, 19093)
After filter genes: (16382, 17379)
Data shape after HVG selection: (16382, 2000)


Finished tracing + transforming fn for pjit in 0.001201629638671875 sec
Finished tracing + transforming _reduce_any for pjit in 0.0009682178497314453 sec
Finished tracing + transforming remainder for pjit in 0.0022249221801757812 sec
Finished tracing + transforming <lambda> for pjit in 0.0011432170867919922 sec
Finished tracing + transforming _is_not_count_val for pjit in 0.014876604080200195 sec
Finished jaxpr to MLIR module conversion jit(_is_not_count_val) in 0.015323638916015625 sec
Finished XLA compilation of jit(_is_not_count_val) in 0.05855607986450195 sec
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Epoch 11/400:   2%|▎         | 10/400 [00:13<08:32,  1.31s/it, loss=432, v_num=1]

### 5.2 Geneformer

Geneformer is a transformer model pretrained on large-scale single-cell transcriptomes. It treats cells as sequences of gene tokens ranked by expression level.


In [ ]:
SAVE_EXT = "loom" # Options: "loom", "h5ad"
MODEL_DIR = os.path.join(MODEL_FOLDER, "Geneformer/default/12L")
DICT_DIR = os.path.join(MODEL_FOLDER, "Geneformer/dicts")
PREPROCESSED_DIR = os.path.join(DATA_FOLDER, f"geneformer/{DATASET_NAME}/{LAYER_KEY}")
if not os.path.exists(PREPROCESSED_DIR):
    os.makedirs(PREPROCESSED_DIR)

In [ ]:
@monitor_inference_resources
def run_geneformer():
    """
    Extract cell embeddings using Geneformer model.
    
    Geneformer treats cells as sequences of gene tokens ranked by expression,
    then uses transformer architecture to learn contextualized representations.
    
    Steps:
    1. Load pretrained Geneformer model and vocabulary
    2. Preprocess data to Geneformer format (gene ranking)
    3. Tokenize cell data into gene sequences
    4. Extract embeddings from transformer layers
    """
    print("🧬 Starting Geneformer cell embedding extraction...")
    
    # Initialize Geneformer instance
    geneform = geneformer_forward.Geneformer_instance(
        save_dir=os.path.join(OUTPUT_DIR, "geneformer"), 
        saved_model_path=MODEL_DIR,
        explicit_save_dir=True,
        num_workers=NUM_WORKERS,
        batch_size=BATCH_SIZE
    )
    
    # Load pretrained model and vocabulary
    geneform.load_pretrained_model()
    geneform.load_vocab(DICT_DIR)

    total_params = calculate_params(geneform.model)
    print(f"📊 Total parameters: {total_params/1e6:.1f} million")
    
    # Prepare data paths
    dataset_name = os.path.basename(ADATA_PATH).split(".")[0]
    processed_adata_path = os.path.join(PREPROCESSED_DIR, f"{dataset_name}.{SAVE_EXT}")
    
    # Preprocess and tokenize data if not already done
    if not os.path.exists(os.path.join(PREPROCESSED_DIR, f"{dataset_name}.dataset")):
        input_data = data.InputData(adata_dataset_path=ADATA_PATH)
        
        if not os.path.exists(processed_adata_path):
            input_data.preprocess_data(
                gene_col=GENE_COL,
                model_type="geneformer",
                save_ext=SAVE_EXT, 
                gene_name_id_dict=geneform.gene_name_id,
                preprocessed_path=PREPROCESSED_DIR,
                counts_layer=LAYER_KEY,
                data_is_raw=DATA_IS_RAW
            )
        
        # Tokenize cells into gene sequences
        geneform.tokenize_data(
            adata_path=processed_adata_path,
            dataset_path=PREPROCESSED_DIR,
            cell_type_col=LABEL_COL,
            data_is_raw=DATA_IS_RAW
        )
    else:
        geneform.load_tokenized_dataset(os.path.join(PREPROCESSED_DIR, f"{DATASET_NAME}.dataset"))
        input_data = data.InputData(adata_dataset_path=processed_adata_path)
    
    print(f"Processed data shape: {input_data.adata.X.shape}")
    
    # Extract embeddings from second-to-last layer (recommended)
    geneform.extract_embeddings(
        data=input_data,
        batch_size=BATCH_SIZE, 
        layer=-2  # Use 2nd to last layer
    )
    
    cell_embeddings = geneform.cell_embeddings
    print(f"Cell embeddings shape: {cell_embeddings.shape}")  
    np.save(os.path.join(OUTPUT_DIR, "geneformer", "cell_emb.npy"), cell_embeddings)
    print("✅ Geneformer embeddings saved successfully")


In [ ]:
run_geneformer()

### 5.3 scGPT

scGPT is a generative pretrained transformer specifically designed for single-cell analysis, using binned gene expression values as input tokens.


In [ ]:
N_BINS = 51
MODEL_DIR = os.path.join(MODEL_FOLDER, "scgpt/scGPT_human")
N_HVG = 1200

In [ ]:
@monitor_inference_resources
def run_scgpt():
    """
    Extract cell embeddings using scGPT model.
    
    scGPT uses a generative transformer architecture with binned gene expression
    values as input tokens, enabling generation and analysis of single-cell data.
    
    Steps:
    1. Initialize scGPT model with configuration
    2. Preprocess data with gene binning
    3. Tokenize binned expression data
    4. Extract cell embeddings from transformer encoder
    """
    print("🧬 Starting scGPT cell embedding extraction...")
    
    # Initialize scGPT instance
    scgpt_model = scgpt_forward.scGPT_instance(
        saved_model_path=MODEL_DIR,
        model_run="pretrained",
        batch_size=BATCH_SIZE, 
        save_dir=os.path.join(OUTPUT_DIR, "scgpt"),
        num_workers=NUM_WORKERS, 
        explicit_save_dir=True
    )
    
    # Create model configuration
    scgpt_model.create_configs(
        seed=SEED, 
        max_seq_len=N_HVG+1,  # +1 for special token
        n_bins=N_BINS
    )
    
    # Load pretrained model
    scgpt_model.load_pretrained_model()
    total_params = calculate_params(scgpt_model.model)
    print(f"📊 Total parameters: {total_params/1e6:.1f} million")
    
    # Prepare input data
    input_data = data.InputData(adata_dataset_path=ADATA_PATH)
    vocab_list = scgpt_model.vocab.get_stoi().keys()
    
    if BATCH_COL is not None:
        input_data.add_batch_labels(batch_key=BATCH_COL)
    
    # Preprocess data with binning for scGPT
    input_data.preprocess_data(
        gene_vocab=vocab_list,
        model_type="scGPT",
        gene_col=GENE_COL,
        data_is_raw=DATA_IS_RAW, 
        normalize_total=NORMALIZE_TOTAL if DATA_IS_RAW else 0,
        counts_layer=LAYER_KEY, 
        n_bins=N_BINS,
        n_hvg=N_HVG
    )

    # Tokenize binned data
    scgpt_model.tokenize_data(
        data=input_data,
        input_layer_key="X_binned",
        include_zero_genes=False
    )
    
    print(f"Processed data shape: {input_data.adata.X.shape}")
    
    # Extract cell embeddings
    scgpt_model.extract_embeddings(data=input_data, save_outputs=False)
    
    cell_embeddings = scgpt_model.cell_embeddings
    print(f"Cell embeddings shape: {cell_embeddings.shape}")
    np.save(os.path.join(OUTPUT_DIR, "scgpt", "cell_emb.npy"), cell_embeddings)
    print("✅ scGPT embeddings saved successfully")

In [ ]:
run_scgpt()

### 5.4 UCE (Universal Cell Embeddings)

UCE generates universal cell embeddings by leveraging protein language models (ESM-2) to create gene embeddings, then aggregating them at the cell level.


In [ ]:
SPECIES = "human"
MODEL_LOC = os.path.join(MODEL_FOLDER, "UCE/33l_8ep_1024t_1280.torch")

In [ ]:
@monitor_inference_resources
def run_uce():
    """
    Extract cell embeddings using UCE (Universal Cell Embeddings).
    
    UCE uses protein language models to create gene-level embeddings,
    then aggregates them to create universal cell representations.
    
    This function calls the UCE evaluation script as a subprocess.
    """
    print("🧬 Starting UCE cell embedding extraction...")
    
    # Change to UCE directory and run evaluation script
    os.chdir("./UCE")
    command = [
        "python3", "eval_single_anndata.py",
        "--adata_path", ADATA_PATH,
        "--dir", os.path.join(OUTPUT_DIR, "uce"),
        "--species", SPECIES,
        "--model_loc", MODEL_LOC,
        "--batch_size", str(BATCH_SIZE),  # Convert to string for subprocess
        "--nlayers", "33",  # Use 33-layer ESM-2 model
        "--layer_key", LAYER_KEY,
        "--gene_col", GENE_COL,
        "--data_is_raw", str(DATA_IS_RAW),
        "--skip", "1"
    ]
    
    print(f"🔧 Running UCE with command: {' '.join(command)}")
    subprocess.run(command)
    print("✅ UCE embeddings extraction completed")

In [ ]:
run_uce()

### 5.5 xTrimoGene

xTrimoGene is a foundation model for single-cell genomics that can handle both single-cell and bulk RNA-seq data with various output types.


In [ ]:
INPUT_TYPE = 'singlecell'
OUTPUT_TYPE = 'cell'
POOL_TYPE = 'all'
TGTHIGHRES = 't4.5'
PRE_NORMALIZED = 'F'
VERSION = 'rde'
MODEL_PATH = os.path.join(MODEL_FOLDER, "scFoundation")

In [ ]:
@monitor_inference_resources
def run_xtrimo():
    """
    Extract cell embeddings using xTrimoGene model.
    
    xTrimoGene requires specific preprocessing to match its expected
    19,264 gene vocabulary and can output different types of embeddings.
    
    Steps:
    1. Load and preprocess data to xTrimoGene format
    2. Select and pad genes to match model vocabulary (19,264 genes)
    3. Save preprocessed data in compressed format
    4. Run xTrimoGene inference via subprocess
    """
    print("🧬 Starting xTrimoGene cell embedding extraction...")
    
    # Load data and extract expression matrix
    adata = sc.read(ADATA_PATH)
    
    if LAYER_KEY == "X":
        if DATA_IS_RAW and adata.raw is not None:
            adata.X = adata.raw.X.copy()
            del adata.raw
            print("Using raw counts from adata.raw.X")
    else:
        adata.X = adata.layers[LAYER_KEY].copy()
        print(f"Using counts from adata.layers[{LAYER_KEY}]")
    
    # Get gene names
    if GENE_COL in adata.var.columns:
        columns = adata.var[GENE_COL].tolist()
    else:
        columns = adata.var.index.tolist()
    
    # Convert to DataFrame
    X_df = pd.DataFrame(
        adata.X.A if sp.issparse(adata.X) else adata.X, 
        index=adata.obs.index.tolist(),
        columns=columns
    )
    
    # Load xTrimoGene gene vocabulary (19,264 genes)
    gene_list_df = pd.read_csv(f'{MODEL_PATH}/OS_scRNA_gene_index.19264.tsv', header=0, delimiter='\t')
    gene_list = list(gene_list_df['gene_name'])
    print(f"📊 xTrimoGene vocabulary: {len(gene_list)} genes")
    
    # Select and pad genes to match vocabulary
    X_df, to_fill_columns, var = main_gene_selection(X_df, gene_list)
    print(f"📊 Padded {len(to_fill_columns)} missing genes with zeros")
    
    # Prepare output directory
    preprocessed_dir = os.path.dirname(ADATA_PATH) + "/xTrimoGene/"
    if not os.path.exists(preprocessed_dir):
        os.makedirs(preprocessed_dir)
    
    # Save preprocessed data
    dataset_name = os.path.basename(ADATA_PATH).split(".")[0]
    preprocessed_path = os.path.join(preprocessed_dir, f"{dataset_name}_19264_{LAYER_KEY}.npz")
    np.savez_compressed(
        preprocessed_path,
        values=X_df.to_numpy(),
        index=X_df.index.to_numpy(),
        columns=X_df.columns.to_numpy()
    )
    print(f"💾 Preprocessed data saved to: {preprocessed_path}")
    
    # Run xTrimoGene inference
    os.chdir("./xTrimoGene/model")
    command = [
        "python3", "get_embedding.py",
        "--task_name", "mapping",
        "--input_type", INPUT_TYPE,
        "--output_type", OUTPUT_TYPE,
        "--pool_type", POOL_TYPE,
        "--tgthighres", TGTHIGHRES,
        "--data_path", preprocessed_path,
        "--save_path", os.path.join(OUTPUT_DIR, "xTrimoGene"),
        "--model_path", MODEL_PATH,
        "--pre_normalized", PRE_NORMALIZED,
        "--version", VERSION
    ]
    
    print(f"🔧 Running xTrimoGene with command: {' '.join(command)}")
    subprocess.run(command)
    print("✅ xTrimoGene embeddings extraction completed")

In [ ]:
run_xtrimo()

### 5.6 LangCell

LangCell combines cellular biology knowledge with language modeling, using both gene expression and text descriptions for enhanced representations.


In [ ]:
NORMALIZE = 0
PASS_CELL_CLS = 0
MODEL_DIR = os.path.join(MODEL_FOLDER, "LangCell")
TOKENIZER_DIR = os.path.join(MODEL_FOLDER, "LangCell/tokenizer/BiomedBERT")
PREPROCESSED_DIR = os.path.join(DATA_FOLDER, f"geneformer/{DATASET_NAME}/{LAYER_KEY}")

In [ ]:
@monitor_inference_resources
def run_langcell():
    """
    Extract cell embeddings using LangCell model.
    
    LangCell integrates biological knowledge with language modeling
    to create enhanced cell representations that consider both
    expression patterns and biological context.
    
    Steps:
    1. Load LangCell model and text tokenizer
    2. Tokenize both expression and text data
    3. Extract embeddings with optional normalization
    """
    print("🧬 Starting LangCell cell embedding extraction...")
    
    # Initialize LangCell instance
    langcell_model = langcell_forward.Langcell_instance(
        saved_model_path=MODEL_DIR,
        saved_tokenizer_path=TOKENIZER_DIR,
        batch_size=BATCH_SIZE,
        save_dir=os.path.join(OUTPUT_DIR, "langcell"),
        num_workers=NUM_WORKERS, 
        explicit_save_dir=True
    )
    
    # Load pretrained model and text tokenizer
    langcell_model.load_pretrained_model()
    langcell_model.load_text_tokenizer()
    
    total_params = calculate_params(langcell_model.model)
    print(f"📊 Total parameters: {total_params/1e6:.1f} million")

    # Prepare data
    dataset_name = os.path.basename(ADATA_PATH).split(".")[0]
    processed_adata_path = os.path.join(PREPROCESSED_DIR, f"{dataset_name}.{SAVE_EXT}")
    input_data = data.InputData(adata_dataset_path=processed_adata_path)

    # Tokenize data (both expression and text)
    langcell_model.tokenize_data(
        adata_path=processed_adata_path,
        dataset_path=PREPROCESSED_DIR,
        cell_type_col=LABEL_COL,
        data_is_raw=DATA_IS_RAW,
        include_zero_genes=False
    )
    langcell_model.get_dataloader()
  
    # Extract embeddings with optional processing
    langcell_model.extract_embeddings(
        data=input_data,
        pass_cell_cls=bool(PASS_CELL_CLS),  # Whether to pass cell classification token
        normalize=bool(NORMALIZE)  # Whether to normalize embeddings
    )
    
    cell_embeddings = langcell_model.cell_embeddings
    print(f"Cell embeddings shape: {cell_embeddings.shape}")
    if PASS_CELL_CLS and NORMALIZE:
        np.save(os.path.join(OUTPUT_DIR, "langcell_passcellcls_normalize", "cell_emb.npy"), cell_embeddings)
    elif NORMALIZE:
        np.save(os.path.join(OUTPUT_DIR, "langcell_normalized", "cell_emb.npy"), cell_embeddings)
    elif PASS_CELL_CLS:
        np.save(os.path.join(OUTPUT_DIR, "langcell_passcellcls", "cell_emb.npy"), cell_embeddings)
    else:
        np.save(os.path.join(OUTPUT_DIR, "langcell", "cell_emb.npy"), cell_embeddings)
    print("✅ LangCell embeddings saved successfully")


In [ ]:
run_langcell()

### 5.7 scBERT

scBERT adapts the BERT architecture for single-cell data by binning gene expression values and using positional embeddings for genes.


In [ ]:
BIN_NUM = 5
GENE_NUM = 16906
MODEL_DIR = os.path.join(MODEL_FOLDER, "scBERT")

In [ ]:
def run_scbert():
    """
    Extract cell embeddings using scBERT model.
    
    scBERT adapts BERT architecture for single-cell data by:
    1. Binning gene expression values into discrete tokens
    2. Using positional embeddings for gene locations
    3. Learning contextualized representations of cells
    
    Steps:
    1. Load and configure scBERT model
    2. Create data loader with expression binning
    3. Extract embeddings from BERT encoder
    """
    print("🧬 Starting scBERT cell embedding extraction...")
    
    adata = sc.read(ADATA_PATH)
    
    # Initialize scBERT instance
    scbert_model = scbert_forward.scBERT_instance(
        saved_model_path=MODEL_DIR,
        batch_size=BATCH_SIZE,
        save_dir=os.path.join(OUTPUT_DIR, "scbert"),
        num_workers=NUM_WORKERS, 
        explicit_save_dir=True
    )
    
    # Create model configuration
    scbert_model.create_configs(
        seed=SEED, 
        gene_num=GENE_NUM,  # Number of genes in vocabulary
        bin_num=BIN_NUM,    # Number of expression bins
        pos_embed_using=True     # Use positional embeddings
    )
    
    # Load pretrained model
    scbert_model.load_pretrained_model()
    total_params = calculate_params(scbert_model.model)
    print(f"📊 Total parameters: {total_params/1e6:.1f} million")

    # Prepare data loader with binning
    scbert_model.get_dataloader(
        adata_path=ADATA_PATH, 
        layer_key=LAYER_KEY,
        gene_col=GENE_COL,
        data_is_raw=bool(DATA_IS_RAW)
    )

    # Extract embeddings
    scbert_model.extract_embeddings(adata)
    cell_embeddings = scbert_model.cell_embeddings
    
    print(f"Cell embeddings shape: {cell_embeddings.shape}")  
    if PASS_CELL_CLS and NORMALIZE:
        np.save(os.path.join(OUTPUT_DIR, "scbert_passcellcls_normalize", "cell_emb.npy"), cell_embeddings)
    elif NORMALIZE:
        np.save(os.path.join(OUTPUT_DIR, "scbert_normalized", "cell_emb.npy"), cell_embeddings)
    elif PASS_CELL_CLS:
        np.save(os.path.join(OUTPUT_DIR, "scbert_passcellcls", "cell_emb.npy"), cell_embeddings)
    else:
        np.save(os.path.join(OUTPUT_DIR, "scbert", "cell_emb.npy"), cell_embeddings)
    print("✅ scBERT embeddings saved successfully")


In [ ]:
run_scbert()

### 5.8 scCello

scCello uses contrastive learning to create cell representations by learning to distinguish between different cell types and states.


In [ ]:
MODEL_DIR = os.path.join(MODEL_FOLDER, "scCello")
PREPROCESSED_DIR = os.path.join(DATA_FOLDER, f"geneformer/{DATASET_NAME}/{LAYER_KEY}")

In [ ]:
def run_sccello():
    """
    Extract cell embeddings using scCello model.
    
    scCello uses contrastive learning to create robust cell representations
    by learning to distinguish between different cell types and states.
    
    Steps:
    1. Load scCello contrastive model
    2. Tokenize expression data
    3. Extract embeddings with optional normalization
    """
    print("🧬 Starting scCello cell embedding extraction...")
    
    # Initialize scCello instance
    sccello_model = sccello_forward.scCello_instance(
        saved_model_path=MODEL_DIR,
        batch_size=BATCH_SIZE,
        save_dir=os.path.join(OUTPUT_DIR,"sccello"),
        num_workers=NUM_WORKERS, 
        explicit_save_dir=True
    )
    
    # Load pretrained model
    sccello_model.load_pretrained_model()
    total_params = calculate_params(sccello_model.model)
    print(f"📊 Total parameters: {total_params/1e6:.1f} million")

    # Prepare data
    dataset_name = os.path.basename(ADATA_PATH).split(".")[0]
    processed_adata_path = os.path.join(PREPROCESSED_DIR, f"{dataset_name}.{SAVE_EXT}")
    input_data = data.InputData(adata_dataset_path=processed_adata_path)
    
    # Tokenize data
    sccello_model.tokenize_data(
        adata_path=processed_adata_path,
        dataset_path=PREPROCESSED_DIR,
        cell_type_col=LABEL_COL,
        data_is_raw=DATA_IS_RAW,
        include_zero_genes=False
    )
    sccello_model.get_dataloader()
    
    # Extract embeddings with contrastive learning features
    sccello_model.extract_embeddings(
        data=input_data,
        pass_cell_cls=bool(PASS_CELL_CLS),  # Use cell classification features
        normalize=bool(NORMALIZE)  # Normalize embeddings
    )
    
    cell_embeddings = sccello_model.cell_embeddings
    print(f"Cell embeddings shape: {cell_embeddings.shape}")  
    np.save(os.path.join(OUTPUT_DIR, "sccello", "cell_emb.npy"), cell_embeddings)
    print("✅ scCello embeddings saved successfully")

In [ ]:
run_sccello()

### 5.9 Harmony

Harmony is a batch correction method that removes technical effects while preserving biological variation in single-cell data.


In [ ]:
N_HVG = 2000

In [ ]:
def run_harmony():
    """
    Extract batch-corrected cell embeddings using Harmony.
    
    Harmony performs batch correction on PCA coordinates to remove
    technical effects while preserving biological variation.
    
    Steps:
    1. Load data and perform standard preprocessing
    2. Compute PCA if not already available
    3. Apply Harmony batch correction
    4. Return corrected embeddings
    """
    print("🧬 Starting Harmony batch correction...")
    
    adata = sc.read(ADATA_PATH)

    # Compute PCA if not already available
    if "X_pca" not in adata.obsm:
        print("📊 Computing PCA coordinates...")
        adata = preprocess_sc_data(adata)
        
        # Select highly variable genes
        sc.pp.highly_variable_genes(
            adata, 
            flavor="seurat", 
            subset=False, 
            n_top_genes=N_HVG, 
            batch_key=BATCH_COL
        )
        adata = adata[:, adata.var.highly_variable].copy()
        
        # Scale data and compute PCA
        sc.pp.scale(adata)
        sc.tl.pca(adata, n_comps=50)

    print(f"📊 Input PCA shape: {adata.obsm['X_pca'].shape}")
    
    # Apply Harmony batch correction
    ho = hm.run_harmony(
        adata.obsm["X_pca"], 
        adata.obs, 
        vars_use=BATCH_COL
    )
    
    # Get corrected embeddings
    cell_embeddings = ho.Z_corr.T
    print(f"Harmony-corrected embeddings shape: {cell_embeddings.shape}")  
    np.save(os.path.join(OUTPUT_DIR, "harmony", "cell_emb.npy"), cell_embeddings)
    print("✅ Harmony-corrected embeddings saved successfully")


def run_scfm_harmony():
    """
    Apply Harmony correction to pre-computed foundation model embeddings.
    
    This function loads existing cell embeddings from foundation models
    and applies Harmony batch correction to them.
    """
    print("🧬 Starting Harmony correction on foundation model embeddings...")
    
    adata = sc.read(ADATA_PATH)
    
    # TODO: Load cell embeddings from foundation models
    # cell_embeddings = load_foundation_model_embeddings()
    
    # Apply Harmony correction
    ho = hm.run_harmony(cell_embeddings, adata.obs, vars_use=BATCH_COL)
    cell_embeddings = ho.Z_corr.T

    print(f"Harmony-corrected foundation model embeddings shape: {cell_embeddings.shape}")
    np.save(os.path.join(OUTPUT_DIR, "harmony", "cell_emb.npy"), cell_embeddings)
    print("✅ Foundation model + Harmony embeddings saved successfully")


In [ ]:
run_harmony()

## 🎉 Summary

This notebook provides a comprehensive framework for extracting cell embeddings from multiple single-cell foundation models:

### 📊 **Supported Models:**
1. **scVI** - Variational autoencoder with batch correction
2. **Geneformer** - Transformer with gene ranking sequences  
3. **scGPT** - Generative transformer with binned expression
4. **UCE** - Universal embeddings using protein language models
5. **xTrimoGene** - Foundation model with 19K gene vocabulary
6. **LangCell** - Biology-aware language modeling
7. **scBERT** - BERT adaptation for single-cell data
8. **scCello** - Contrastive learning approach
9. **Harmony** - Batch correction and integration

### 📁 **Output:**
- Cell embeddings saved as numpy arrays (`cell_emb.npy`)
- Model-specific preprocessing artifacts
- Execution logs for debugging and monitoring

### 🚀 **Usage:**
1. Configure your dataset and model paths
2. Choose your target foundation model
3. Set preprocessing parameters
4. Run the extraction function
5. Use embeddings for downstream analysis (clustering, classification, visualization)

The notebook is designed to be modular and extensible, allowing easy addition of new models and customization of preprocessing pipelines.